In [1]:
import os, random
import numpy as np
import torch
from local_negotiator_v2 import (
    LocalNegotiatorGNN,
    evaluate_strict_decentralized,
    load_negotiator_dataset,
    prepare_dataset,
)
from decent_auction import (
    DecentAuctionModel,
    evaluate_decent_auction,
    train_decent_auction_model,
    DECENT_AUCTION_ROUNDS,
)

SEED = 42
DATASET = './dataset/negotiator_dataset_v1.pt'
BASE_CKPT = 'strict_local_negotiator_best_v1.pt'
OUT = './model/decent_auction_best.pt'
EPOCHS = 80
LR = 3e-4
FREEZE_EPOCHS = 10
MAX_TRAIN = 7000
MAX_VAL = 500
MAX_TEST = 500

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | Torch: {torch.__version__}')


Device: cuda | Torch: 2.10.0+cu128


In [2]:
base = LocalNegotiatorGNN().to(DEVICE)
if os.path.isfile(BASE_CKPT):
    ckpt = torch.load(BASE_CKPT, map_location='cpu', weights_only=False)
    base.load_state_dict(ckpt['model_state_dict'])
    base.to(DEVICE)
    print('Loaded base checkpoint:')
    print(f"  gossip bijection : {ckpt['metrics']['test']['bijection_rate']:.3f}")
    print(f"  gossip cost ratio: {ckpt['metrics']['test']['cost_ratio_vs_hungarian']:.4f}")
    print(f"  gossip match     : {ckpt['metrics']['test']['slot_match_rate']:.3f}")
else:
    print('No base checkpoint; training DecentAuction from scratch')

model = DecentAuctionModel(base).to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
new_p = sum(p.numel() for p in model.gnn.price_proj.parameters())
print(f'\nTotal params : {total_p:,}')
print(f'New params   : {new_p:,} (frozen base: {total_p - new_p:,})')


Loaded base checkpoint:
  gossip bijection : 0.589
  gossip cost ratio: 1.0340
  gossip match     : 0.640

Total params : 32,610
New params   : 1,984 (frozen base: 30,626)


In [3]:
raw = load_negotiator_dataset(DATASET)
train_data, val_data, test_data = prepare_dataset(
    raw,
    model.formation_embedding.weight.detach().cpu(),
    seed=SEED,
    force_gt_visibility=False,
)
train_data = train_data[:MAX_TRAIN]
val_data = val_data[:MAX_VAL]
test_data = test_data[:MAX_TEST]
print(f'Train={len(train_data)} Val={len(val_data)} Test={len(test_data)}')


Train=7000 Val=500 Test=500


In [4]:
print('=== Baselines before DecentAuction training ===')
print('\nGossip consensus (base model):')
gossip = evaluate_strict_decentralized(base, test_data, DEVICE)
for k, v in gossip.items(): print(f'  {k}: {v:.4f}')

print('\nDecentAuction (local-state auction):')
baseline = evaluate_decent_auction(model, test_data[:200], DEVICE)
for k, v in baseline.items(): print(f'  {k}: {v:.4f}')


=== Baselines before DecentAuction training ===

Gossip consensus (base model):
  bijection_rate: 0.8360
  conflict_rate: 0.0024
  unassigned_rate: 0.0083
  cost_ratio_vs_hungarian: 1.0265
  slot_match_rate: 0.6785
  consensus_rounds: 8.6480
  converged_rate: 0.7200

DecentAuction (local-state auction):
  bijection_rate: 0.8050
  conflict_rate: 0.0013
  unassigned_rate: 0.0116
  cost_ratio_vs_hungarian: 1.0326
  slot_match_rate: 0.8038
  consensus_rounds: 11.1200
  converged_rate: 0.7950
  messages_sent: 1558.9500


In [5]:
print('=== DecentAuction training ===')
history = train_decent_auction_model(
    model,
    train_data,
    val_data,
    DEVICE,
    epochs=EPOCHS,
    lr=LR,
    freeze_epochs=FREEZE_EPOCHS,
    force_gt_train=True,
)


=== DecentAuction training ===
epoch 001 [frozen|K=0] loss=5.0455 val_cost=1.0663 val_bij=0.640 val_match=0.670 val_conv=0.600
epoch 002 [frozen|K=0] loss=3.8953 val_cost=1.0593 val_bij=0.620 val_match=0.653 val_conv=0.590
epoch 003 [frozen|K=0] loss=3.5386 val_cost=1.0470 val_bij=0.670 val_match=0.697 val_conv=0.610
epoch 004 [frozen|K=0] loss=3.2803 val_cost=1.0430 val_bij=0.740 val_match=0.665 val_conv=0.700
epoch 005 [frozen|K=0] loss=3.0991 val_cost=1.0506 val_bij=0.700 val_match=0.657 val_conv=0.670
epoch 006 [frozen|K=0] loss=2.9821 val_cost=1.0552 val_bij=0.700 val_match=0.690 val_conv=0.690
epoch 007 [frozen|K=0] loss=2.8890 val_cost=1.0495 val_bij=0.730 val_match=0.686 val_conv=0.690
epoch 008 [frozen|K=0] loss=2.8025 val_cost=1.0515 val_bij=0.700 val_match=0.682 val_conv=0.680
epoch 009 [frozen|K=0] loss=2.6952 val_cost=1.0419 val_bij=0.780 val_match=0.697 val_conv=0.700
  [Epoch 10] Encoder unfrozen (bijection=0.750, guard=0.450).
epoch 010 [joint|K=0] loss=2.5992 val_cost=

c:\Users\ASUS\Desktop\gnn_project\2_models_approach\decent_auction.py:748: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  joint_scheduler.step()


epoch 011 [joint|K=0] loss=1.5219 val_cost=1.0434 val_bij=0.730 val_match=0.727 val_conv=0.690
epoch 012 [joint|K=0] loss=1.2180 val_cost=1.0335 val_bij=0.830 val_match=0.750 val_conv=0.790
epoch 013 [joint|K=1] loss=1.1147 val_cost=1.0210 val_bij=0.850 val_match=0.760 val_conv=0.820
epoch 014 [joint|K=1] loss=1.0487 val_cost=1.0325 val_bij=0.850 val_match=0.768 val_conv=0.820
epoch 015 [joint|K=1] loss=1.0048 val_cost=1.0247 val_bij=0.860 val_match=0.754 val_conv=0.840
epoch 016 [joint|K=1] loss=0.9763 val_cost=1.0220 val_bij=0.840 val_match=0.754 val_conv=0.810
epoch 017 [joint|K=1] loss=0.9529 val_cost=1.0193 val_bij=0.880 val_match=0.775 val_conv=0.870
epoch 018 [joint|K=2] loss=0.9386 val_cost=1.0154 val_bij=0.910 val_match=0.785 val_conv=0.860
epoch 019 [joint|K=2] loss=0.9235 val_cost=1.0176 val_bij=0.890 val_match=0.772 val_conv=0.870
epoch 020 [joint|K=2] loss=0.9112 val_cost=1.0214 val_bij=0.850 val_match=0.791 val_conv=0.820
epoch 021 [joint|K=2] loss=0.8981 val_cost=1.0157 

KeyboardInterrupt: 

In [7]:
print('=== Final evaluation (test set) ===')
final = evaluate_decent_auction(model, test_data, DEVICE)
gossip_final = evaluate_strict_decentralized(base, test_data, DEVICE)
keys = ['bijection_rate', 'cost_ratio_vs_hungarian', 'slot_match_rate',
        'conflict_rate', 'unassigned_rate', 'consensus_rounds', 'messages_sent']
print(f'{"metric":<35} {"gossip (base)":>16} {"decent":>12}')
print('-' * 65)
for k in keys:
    g = gossip_final.get(k, 0.0)
    d = final.get(k, 0.0)
    better = ''
    if k in ('bijection_rate', 'slot_match_rate') and d > g: better = ' up'
    if k in ('cost_ratio_vs_hungarian', 'conflict_rate', 'unassigned_rate',
             'consensus_rounds', 'messages_sent') and d < g: better = ' down'
    print(f'{k:<35} {g:>16.4f} {d:>12.4f}{better}')


=== Final evaluation (test set) ===
metric                                 gossip (base)       decent
-----------------------------------------------------------------
bijection_rate                                0.8760       0.8620
cost_ratio_vs_hungarian                       1.0241       1.0185 down
slot_match_rate                               0.6492       0.7715 up
conflict_rate                                 0.0024       0.0030
unassigned_rate                               0.0058       0.0069
consensus_rounds                              8.6920      10.0520
messages_sent                                 0.0000    1385.5000


In [9]:
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'base_checkpoint': BASE_CKPT,
        'epochs': EPOCHS,
        'lr': LR,
        'freeze_epochs': FREEZE_EPOCHS,
        'auction_rounds': DECENT_AUCTION_ROUNDS,
        'inference': 'fully decentralized local-state auction; no Hungarian/Sinkhorn/global price vector',
    },
    'metrics': {
        'decent_auction': final,
        'gossip_baseline': gossip_final,
    },
}, OUT)
print(f'Saved -> {OUT}')


Saved -> ./model/decent_auction_best.pt


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('DecentAuction training', fontsize=13, fontweight='bold')
ep = range(1, len(history['train_loss']) + 1)
axes[0].plot(ep, history['train_loss'], label='train', color='steelblue')
axes[0].plot(ep, history['val_loss'], label='val', color='coral', linestyle='--')
axes[0].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.7, label='unfreeze')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history['val_bijection_rate'], label='bijection', color='green')
axes[1].plot(ep, history['val_slot_match_rate'], label='match', color='purple', linestyle='--')
axes[1].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.7)
axes[1].set_title('Bijection & Match Rate'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(ep, history['val_cost_ratio_vs_hungarian'], label='cost ratio', color='orange')
axes[2].axhline(1.0, color='gray', linestyle='-', alpha=0.3, label='optimal')
axes[2].set_title('Cost Ratio vs Hungarian'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('decent_auction_training_curves.png', dpi=120)
plt.show()
print('Saved decent_auction_training_curves.png')
